# Latih detektor penyakit strawberry (YOLO26) di Kaggle

Notebook ini menggantikan `train.py`. `realtime.py` **tidak** dijalankan di sini: Kaggle tidak punya kamera maupun layar, dan berjalan di server cloud sehingga tidak bisa menjangkau ESP32-CAM di jaringan Anda. Realtime tetap dijalankan di laptop (VSCode) memakai `best.pt` dari notebook ini.

**Atur dulu di panel Settings (sisi kanan), sebelum menjalankan sel apa pun:**

1. **Accelerator**: GPU T4 x2. Skrip hanya memakai satu GPU dan itu sudah cukup. Kalau muncul error CUDA "no kernel image", ganti ke T4.
2. **Internet**: On. Diperlukan untuk `pip install` dan mengunduh bobot awal `yolo26s.pt`. GPU dan internet di Kaggle mengharuskan akun terverifikasi nomor telepon.
3. **Add Input**, tiga dataset:
   - dataset strawberry pertama (**wajib**: gambar penyakit beserta labelnya);
   - `strawberry_extra` (paket negatif lama yang sudah Anda unggah, tidak perlu diganti);
   - dataset baru berisi `strawberry_extra_v2.zip` **dan** `benchmark_healthy.zip`: Create → New Dataset, unggah kedua zip sekaligus. Isinya 300 negatif strawberry matang/mentah, `split_frozen.csv` (pembagian tetap), dan 562 foto patokan strawberry sehat.

`best.zip` / `best.pt` adalah bobot model, bukan dataset, jadi tidak perlu dilampirkan (notebook inilah yang menghasilkannya).

Sel 2b membagi train/val/test supaya jepretan dari objek yang sama tidak tersebar di beberapa split (sekitar 30 detik). Dengan `split_frozen.csv` dari `strawberry_extra_v2`, test set gambar penyakit sama persis dengan run 2, sehingga hasil antar run bisa dibandingkan langsung.

Lalu jalankan sel dari atas ke bawah. Untuk training panjang, pakai **Save Version → Save & Run All** supaya tetap berjalan walau tab ditutup.

In [ ]:
# 1) Pasang Ultralytics dan cek GPU
# Jika muncul error saat import setelah pip install, restart kernel lalu lanjut dari sel ini.
!pip install -q -U ultralytics

import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU TIDAK aktif -> Settings > Accelerator > pilih GPU, lalu jalankan ulang dari sel ini.")

In [ ]:
# 2) Cari dataset: boleh satu folder gabungan, atau beberapa (dataset pertama + paket tambahan)
import os
from pathlib import Path
import yaml

# Urutan = ID kelas di file label (sudah dicocokkan dengan nama file). Jangan diubah.
CLASS_NAMES = [
    "angular_leafspot",       # 0
    "anthracnose_fruit_rot",  # 1
    "blossom_blight",         # 2
    "gray_mold",              # 3
    "leaf_spot",              # 4
    "powdery_mildew_fruit",   # 5
    "powdery_mildew_leaf",    # 6
]

def read_names(root):
    f = root / "data.yaml"
    if not f.is_file():
        return None
    names = yaml.safe_load(f.read_text(encoding="utf-8")).get("names")
    return [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)

def find_dataset_roots(bases=("/kaggle/input", ".")):
    found = set()
    for base in bases:
        for dirpath, _, _ in os.walk(base, followlinks=True):
            root = Path(dirpath)
            if all((root / split / "images").is_dir() for split in ("train", "val", "test")):
                found.add(root.resolve())
    roots, names = [], list(CLASS_NAMES)
    for root in sorted(found):
        n = read_names(root)
        if n is not None and n[: len(CLASS_NAMES)] != CLASS_NAMES:
            print(f"DILEWATI (kelas tidak cocok dengan 7 kelas penyakit): {root} -> {n}")
            continue
        if n is not None and len(n) > len(names):
            names = n  # paket tambahan yang menambah kelas (mis. strawberry_leaf)
        roots.append(root)
    if not roots:
        raise FileNotFoundError("Dataset tidak ditemukan. Tambahkan dataset lewat Add Input, lalu jalankan ulang sel ini.")
    return roots, names

DATA_ROOTS, CLASS_NAMES = find_dataset_roots()
print(f"{len(DATA_ROOTS)} dataset dipakai | {len(CLASS_NAMES)} kelas: {CLASS_NAMES}")
def count_labels(root):
    files = list((root / "train" / "labels").glob("*.txt"))
    pos = sum(1 for f in files if f.stat().st_size > 0)
    return pos, len(files) - pos  # (berlabel penyakit, negatif/label kosong)

for root in DATA_ROOTS:
    sizes = {s: sum(1 for _ in (root / s / "images").iterdir()) for s in ("train", "val", "test")}
    pos, neg = count_labels(root)
    print(f"  {root}: {sizes} | train: {pos} berlabel, {neg} negatif")

if sum(count_labels(r)[0] for r in DATA_ROOTS) == 0:
    raise FileNotFoundError(
        "Tidak ada gambar berlabel penyakit. Tambahkan DATASET PERTAMA (gambar penyakit + labelnya) lewat Add Input; "
        "paket strawberry_extra saja hanya berisi gambar negatif."
    )

In [ ]:
# 2b) Bagi ulang train/val/test tanpa gambar kembar (disarankan)
# Dataset ini banyak berisi jepretan beruntun dari daun/bunga yang sama. Pembagian bawaannya acak per gambar,
# sehingga jepretan yang hampir identik tersebar di train dan val/test, dan skor val/test jadi terlalu optimis.
# Sel ini mengelompokkan jepretan dari objek yang sama, lalu membagi per KELOMPOK dengan proporsi tiap kelas dijaga.
# Jika paket strawberry_extra berisi split_frozen.csv, pembagiannya dipakai apa adanya supaya test set sama di setiap run;
# hanya gambar yang belum tercatat yang dibagi otomatis.
import collections, csv, time
import numpy as np
from PIL import Image

CLEAN_SPLIT  = True
FREEZE_SPLIT = True  # pakai split_frozen.csv jika ada (disarankan, agar hasil antar run bisa dibandingkan)
SPLIT_RATIOS = {"train": 0.60, "val": 0.15, "test": 0.25}
SPLIT_SEED   = 0
DUP_ANY      = 40   # jarak hash (dari 256 bit) <= ini: dianggap foto yang sama
DUP_SAME_CLS = 60   # <= ini dan kelasnya sama: dianggap jepretan beruntun objek yang sama
DUP_NO_LABEL = 20   # ambang ketat bila salah satu gambar tanpa label (gambar negatif berlatar polos mudah tampak mirip)
SPLIT_NAMES  = ("train", "val", "test")
IMG_EXT      = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def label_of(img):
    return img.parent.parent / "labels" / (img.stem + ".txt")

def classes_of(img):
    lp = label_of(img)
    if not lp.is_file():
        return frozenset()
    return frozenset(int(float(l.split()[0])) for l in lp.read_text(encoding="utf-8").splitlines() if l.strip())

def original_splits(roots):
    return {s: [p for r in roots for p in sorted((r / s / "images").iterdir()) if p.suffix.lower() in IMG_EXT] for s in SPLIT_NAMES}

_TR = getattr(Image, "Transpose", Image)
_TRANS = [None, _TR.FLIP_LEFT_RIGHT, _TR.FLIP_TOP_BOTTOM, _TR.ROTATE_90, _TR.ROTATE_180, _TR.ROTATE_270, _TR.TRANSPOSE, _TR.TRANSVERSE]

def hash8(path):
    """dHash 256-bit dari 8 orientasi (asli, cermin, rotasi) supaya versi yang dibalik/diputar ikut terdeteksi."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((640, 640))
        out = []
        for t in _TRANS:
            a = np.asarray((im if t is None else im.transpose(t)).convert("L").resize((17, 16), Image.BILINEAR), dtype=np.int16)
            out.append((a[:, 1:] > a[:, :-1]).ravel())
    return np.stack(out)

def group_duplicates(paths, cls_sets):
    n = len(paths)
    t0 = time.time()
    H = np.stack([hash8(p) for p in paths]).astype(np.float32) * 2 - 1   # (n, 8, 256), nilai +-1
    base = H[:, 0]
    D = np.full((n, n), 256, dtype=np.float32)
    for t in range(H.shape[1]):
        np.minimum(D, (256 - H[:, t] @ base.T) / 2, out=D)                # jarak Hamming terkecil antar orientasi
    D = np.minimum(D, D.T)
    print(f"Hash {n} gambar: {time.time() - t0:.0f} detik")
    labeled = np.array([bool(c) for c in cls_sets])
    both = labeled[:, None] & labeled[None, :]
    M = np.zeros((n, len(CLASS_NAMES)), dtype=np.float32)
    for i, c in enumerate(cls_sets):
        M[i, list(c)] = 1
    share = (M @ M.T) > 0                                              # dua gambar punya kelas yang sama
    link = np.where(both, (D <= DUP_ANY) | ((D <= DUP_SAME_CLS) & share), D <= DUP_NO_LABEL)
    link = np.triu(link, 1)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(*np.nonzero(link)):
        parent[find(a)] = find(b)
    groups = collections.defaultdict(list)
    for i in range(n):
        groups[find(i)].append(i)
    return list(groups.values())

def stratified_group_split(groups, cls_sets, nc, ratios, seed):
    """Membagi kelompok ke train/val/test; kelompok berisi kelas paling langka dibagi lebih dulu."""
    K = nc + 1                                             # indeks terakhir = gambar negatif (tanpa label)
    vec = []
    for g in groups:
        v = np.zeros(K)
        for i in g:
            for k in (cls_sets[i] or {nc}):
                v[k] += 1
        vec.append(v)
    total = np.sum(vec, axis=0)
    target = {s: ratios[s] * total for s in SPLIT_NAMES}
    count = {s: np.zeros(K) for s in SPLIT_NAMES}
    n_img = {s: 0 for s in SPLIT_NAMES}
    n_all = sum(len(g) for g in groups)
    rng = np.random.default_rng(seed)
    tie = rng.random(len(groups))
    order = sorted(range(len(groups)), key=lambda j: (min(total[k] for k in np.nonzero(vec[j])[0]), -len(groups[j]), tie[j]))
    assign = {}
    for j in order:
        v = vec[j]
        def score(s):
            need = sum(v[k] * (target[s][k] - count[s][k]) / max(target[s][k], 1e-9) for k in np.nonzero(v)[0])
            return (need, ratios[s] * n_all - n_img[s])
        s = max(SPLIT_NAMES, key=score)
        count[s] += v
        n_img[s] += len(groups[j])
        for i in groups[j]:
            assign[i] = s
    return assign

ORIG = original_splits(DATA_ROOTS)
if CLEAN_SPLIT:
    paths = [p for s in SPLIT_NAMES for p in ORIG[s]]
    orig_of = {p: s for s in SPLIT_NAMES for p in ORIG[s]}
    cls_sets = [classes_of(p) for p in paths]
    groups = group_duplicates(paths, cls_sets)
    multi = [g for g in groups if len(g) > 1]
    leak = [g for g in multi if len({orig_of[paths[i]] for i in g}) > 1]
    leak_val = sum(1 for g in leak for i in g if orig_of[paths[i]] == "val" and any(orig_of[paths[x]] == "train" for x in g))
    leak_test = sum(1 for g in leak for i in g if orig_of[paths[i]] == "test" and any(orig_of[paths[x]] == "train" for x in g))
    n_val, n_test = len(ORIG["val"]), len(ORIG["test"])
    print(f"Kelompok berisi >1 gambar: {len(multi)} ({sum(map(len, multi))} gambar, terbesar {max(map(len, groups))})")
    print(f"Pembagian bawaan: {leak_val} dari {n_val} gambar val ({leak_val / n_val:.0%}) dan "
          f"{leak_test} dari {n_test} gambar test ({leak_test / n_test:.0%}) punya kembaran di train")

    FROZEN = {}
    if FREEZE_SPLIT:
        for r in DATA_ROOTS:
            f = r / "split_frozen.csv"
            if f.is_file():
                with open(f, newline="", encoding="utf-8") as fh:
                    FROZEN.update({row["file"]: row["split"] for row in csv.DictReader(fh)})
    if FROZEN:
        dup_names = [n for n, c in collections.Counter(p.name for p in paths).items() if c > 1]
        if dup_names:
            print(f"PERINGATAN: {len(dup_names)} nama file dipakai lebih dari satu gambar, mis. {dup_names[:3]}")
        assign, free, conflict = {}, [], 0
        for g in groups:
            known = [FROZEN[paths[i].name] for i in g if paths[i].name in FROZEN]
            if known:
                s = collections.Counter(known).most_common(1)[0][0]
                conflict += len(set(known)) > 1
                for i in g:
                    assign[i] = s
            else:
                free.append(g)
        if free:
            assign.update(stratified_group_split(free, cls_sets, len(CLASS_NAMES), SPLIT_RATIOS, SPLIT_SEED))
        n_known = sum(1 for p in paths if p.name in FROZEN)
        print(f"Pembagian tetap (split_frozen.csv): {n_known} gambar | gambar baru yang dibagi otomatis: {sum(map(len, free))}"
              + (f" | PERINGATAN: {conflict} kelompok berisi split berbeda" if conflict else ""))
    else:
        assign = stratified_group_split(groups, cls_sets, len(CLASS_NAMES), SPLIT_RATIOS, SPLIT_SEED)
    SPLITS = {s: sorted(paths[i] for i in range(len(paths)) if assign[i] == s) for s in SPLIT_NAMES}
    still = sum(1 for g in groups if len({assign[i] for i in g}) > 1)
    assert still == 0, "masih ada kelompok yang terbagi ke beberapa split"

    names = CLASS_NAMES + ["(negatif)"]
    def per_class(split_paths):
        c = collections.Counter()
        for p in split_paths:
            for k in (classes_of(p) or {len(CLASS_NAMES)}):
                c[k] += 1
        return c
    old = {s: per_class(ORIG[s]) for s in SPLIT_NAMES}
    new = {s: per_class(SPLITS[s]) for s in SPLIT_NAMES}
    print(f"\n{'jumlah gambar per kelas':24s} {'train':>11s} {'val':>9s} {'test':>9s}   (lama -> baru)")
    for k, name in enumerate(names):
        print(f"{name:24s} " + " ".join(f"{old[s][k]:4d}->{new[s][k]:<4d}" for s in SPLIT_NAMES))
    print(f"{'TOTAL':24s} " + " ".join(f"{len(ORIG[s]):4d}->{len(SPLITS[s]):<4d}" for s in SPLIT_NAMES))

    with open("split_assignment.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["image", "split_lama", "split_baru", "kelompok"])
        gid = {i: n for n, g in enumerate(groups) for i in g}
        for i, p in enumerate(paths):
            w.writerow([str(p), orig_of[p], assign[i], gid[i]])
    print("\nPembagian disimpan di split_assignment.csv. Hasilnya selalu sama selama data, split_frozen.csv dan SPLIT_SEED sama.")
else:
    SPLITS = ORIG
    print("CLEAN_SPLIT = False: memakai pembagian bawaan dataset.")


In [ ]:
# 3) Pengaturan + data.yaml
import math, random

MODEL   = "yolo26s.pt"  # yolo26n.pt = paling ringan, yolo26m.pt = lebih akurat
EPOCHS  = 100
IMGSZ   = 640           # gambar asli 419 px; 416 lebih cepat
BATCH   = 16
WORKERS = 4             # turunkan ke 2 jika muncul error DataLoader / shared memory
RUN     = "strawberry"

# --- Percobaan peningkatan akurasi: ubah SATU per run agar hasilnya bisa dibandingkan ---
USE_RFS  = True   # ulang gambar yang memuat kelas langka (repeat factor sampling)
RFS_T    = 0.2    # makin besar = makin banyak pengulangan kelas langka
CLS_GAIN = 0.5    # bobot loss klasifikasi (default 0.5); coba 1.0 kalau antar kelas sering tertukar

def build_rfs_list(train_paths, nc, t, seed=0):
    """Repeat factor sampling: gambar yang memuat kelas langka diulang agar kelas itu lebih sering dilihat model."""
    imgs = [(p, set(classes_of(p))) for p in train_paths]
    labeled = [c for _, c in imgs if c]  # hanya gambar berlabel, agar jumlah gambar negatif tidak mengubah faktor pengulangan
    freq = [sum(k in c for c in labeled) / len(labeled) for k in range(nc)]  # fraksi gambar berlabel yang memuat kelas k
    rep = [max(1.0, math.sqrt(t / f)) if f > 0 else 1.0 for f in freq]
    rng = random.Random(seed)
    lines = []
    for p, c in imgs:
        rf = max((rep[k] for k in c), default=1.0)
        lines += [str(p)] * (int(rf) + (rng.random() < rf - int(rf)))  # pembulatan stokastik
    return lines, len(imgs), freq, rep

def write_list(name, lines):
    f = Path(name).resolve()
    f.write_text("\n".join(map(str, lines)) + "\n", encoding="utf-8")
    return str(f)

if USE_RFS:
    lines, n_imgs, freq, rep = build_rfs_list(SPLITS["train"], len(CLASS_NAMES), RFS_T)
    print(f"RFS: {n_imgs} gambar train -> {len(lines)} entri per epoch")
    for k, name in enumerate(CLASS_NAMES):
        print(f"  {name:24s} ada di {freq[k] * 100:5.1f}% gambar berlabel | diulang {rep[k]:.2f}x")
else:
    lines = SPLITS["train"]
N_TRAIN = len(lines)

cfg = {
    "train": write_list("train_list.txt", lines),
    "val": write_list("val_list.txt", SPLITS["val"]),
    "test": write_list("test_list.txt", SPLITS["test"]),
    "names": dict(enumerate(CLASS_NAMES)),
}
print({s: len(SPLITS[s]) for s in SPLIT_NAMES}, "gambar | train per epoch:", N_TRAIN)
DATA_YAML = Path("data.yaml").resolve()
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text())

In [ ]:
# 4) Training. Dataset di /kaggle/input bersifat hanya-baca, jadi peringatan "cache not saved" boleh diabaikan.
import os
os.environ["WANDB_MODE"] = "disabled"  # cegah Kaggle meminta API key Weights & Biases

from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    cache=True,    # simpan gambar di RAM agar epoch lebih cepat (dataset kecil)
    patience=30,   # berhenti lebih awal kalau val tidak membaik 30 epoch
    flipud=0.5,    # daun/buah tidak punya orientasi atas-bawah yang tetap
    cls=CLS_GAIN,  # bobot loss klasifikasi
    name=RUN,
)
BEST = Path(model.trainer.best)
print("Bobot terbaik:", BEST)

In [ ]:
# 5) Evaluasi akhir pada split test (jangan dipakai untuk tuning; lihat hanya setelah training selesai)
best = YOLO(str(BEST))
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH)
print(f"\n[TEST] mAP50-95 = {metrics.box.map:.3f} | mAP50 = {metrics.box.map50:.3f} | P = {metrics.box.mp:.3f} | R = {metrics.box.mr:.3f}")
print(f"{'kelas':24s} {'P':>6s} {'R':>6s} {'AP50':>6s} {'AP50-95':>8s}")
PER_CLASS = {}
for k, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(k)
    PER_CLASS[CLASS_NAMES[int(c)]] = {"P": round(float(p), 4), "R": round(float(r), 4), "AP50": round(float(ap50), 4), "AP50_95": round(float(ap), 4)}
    print(f"{CLASS_NAMES[int(c)]:24s} {p:6.3f} {r:6.3f} {ap50:6.3f} {ap:8.3f}")

In [ ]:
# 6) Alarm palsu: berapa gambar negatif (neg_*) di split test yang tetap memicu deteksi penyakit?
import collections
import matplotlib.pyplot as plt

def disease_hit(result):
    """True jika ada deteksi penyakit (kelas strawberry_leaf, jika dipakai, bukan penyakit)."""
    return any(CLASS_NAMES[int(c)] != "strawberry_leaf" for c in result.boxes.cls)

FP_RATE = None
neg = sorted(p for p in SPLITS["test"] if p.name.startswith("neg_"))
if not neg:
    print("Tidak ada gambar negatif (neg_*) di split test. Tambahkan paket strawberry_extra untuk mengukur alarm palsu.")
else:
    CONF = 0.35  # sama dengan default realtime.py
    preds = best.predict([str(p) for p in neg], imgsz=IMGSZ, conf=CONF, verbose=False)
    hits = [(p, r) for p, r in zip(neg, preds) if disease_hit(r)]
    FP_RATE = len(hits) / len(neg)
    print(f"Gambar negatif di test: {len(neg)} | memicu deteksi penyakit (conf >= {CONF}): {len(hits)} ({len(hits) / len(neg) * 100:.1f}%)")
    per_src = collections.defaultdict(lambda: [0, 0])
    for p, r in zip(neg, preds):
        s = per_src[p.name.split("_")[1]]  # neg_<sumber>_0001.jpg
        s[0] += 1
        s[1] += int(disease_hit(r))
    for src, (n, h) in sorted(per_src.items()):
        print(f"  {src:12s}: {h} dari {n}")
    if hits:
        fig, axes = plt.subplots(2, 4, figsize=(18, 9))
        for ax in axes.ravel():
            ax.axis("off")
        for ax, (p, r) in zip(axes.ravel(), hits[:8]):
            ax.imshow(r.plot()[:, :, ::-1])
            ax.set_title(p.name, fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
# 7) Alarm palsu pada strawberry SEHAT: patokan tetap berisi foto matang/mentah yang tidak pernah dipakai training.
#    Pakai angka ini untuk membandingkan run. Acuan pada 562 foto yang sama (conf >= 0.35): run 1 = 2,3%, run 2 = 4,1%.
import collections, os
import matplotlib.pyplot as plt

BENCH_FP, BENCH_N = None, 0
def find_bench_dirs(bases=("/kaggle/input", ".")):
    found = set()
    for base in [*bases, *map(str, DATA_ROOTS)]:
        for dirpath, _, _ in os.walk(base, followlinks=True):
            d = Path(dirpath)
            if d.name == "images" and d.parent.name == "benchmark_healthy":
                found.add(d.resolve())
    return sorted(found)

bench_dirs = find_bench_dirs()
if not bench_dirs:
    print("Folder benchmark_healthy tidak ditemukan. Lampirkan dataset berisi benchmark_healthy.zip (lihat petunjuk di atas).")
else:
    bench = sorted({p.name: p for d in bench_dirs for p in d.iterdir() if p.suffix.lower() in IMG_EXT}.values())
    BENCH_N = len(bench)
    THRESH = (0.35, 0.5, 0.6, 0.7)
    top = []  # (conf tertinggi untuk kelas penyakit, nama kelas) per foto
    for i in range(0, len(bench), 32):
        for r in best.predict([str(p) for p in bench[i:i + 32]], imgsz=IMGSZ, conf=0.2, verbose=False):
            dets = [(float(c), CLASS_NAMES[int(k)]) for k, c in zip(r.boxes.cls, r.boxes.conf)
                    if CLASS_NAMES[int(k)] != "strawberry_leaf"]
            top.append(max(dets, default=(0.0, None)))
    BENCH_FP = {str(t): round(sum(c >= t for c, _ in top) / BENCH_N, 4) for t in THRESH}
    print(f"Patokan: {BENCH_N} foto strawberry sehat")
    for t in THRESH:
        n = sum(c >= t for c, _ in top)
        print(f"  conf >= {t}: {n} foto memicu deteksi penyakit ({n / BENCH_N:.1%})")
    print("  kelas pemicu (conf >= 0.35):", dict(collections.Counter(k for c, k in top if c >= 0.35).most_common()))
    worst = [i for i in sorted(range(BENCH_N), key=lambda i: -top[i][0])[:8] if top[i][0] >= 0.35]
    if worst:
        res = best.predict([str(bench[i]) for i in worst], imgsz=IMGSZ, conf=0.35, verbose=False)
        fig, axes = plt.subplots(2, 4, figsize=(18, 9))
        for ax in axes.ravel():
            ax.axis("off")
        for ax, i, r in zip(axes.ravel(), worst, res):
            ax.imshow(r.plot()[:, :, ::-1])
            ax.set_title(bench[i].name, fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
# 8) Ringkasan run: simpan lalu bandingkan antar percobaan (unduh run_summary.json dari tiap run)
import json

summary = {
    "clean_split": CLEAN_SPLIT, "split_seed": SPLIT_SEED, "split_sizes": {s: len(SPLITS[s]) for s in SPLIT_NAMES},
    "run": RUN, "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
    "use_rfs": USE_RFS, "rfs_t": RFS_T, "cls_gain": CLS_GAIN, "train_entries_per_epoch": N_TRAIN,
    "test_map50": round(float(metrics.box.map50), 4), "test_map50_95": round(float(metrics.box.map), 4),
    "per_class": PER_CLASS, "false_alarm_rate_on_negatives": FP_RATE,
    "healthy_benchmark_n": BENCH_N, "healthy_benchmark_fp": BENCH_FP,
}
Path("run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

# Pembanding: run 2 diuji ulang pada test set yang SAMA (dengan strawberry_extra_v2 dan split_frozen.csv-nya).
# Hanya berlaku selama data dan split_frozen.csv tidak berubah.
REF = {"nama": "run 2", "test_map50": 0.870, "test_map50_95": 0.706, "AP50": {"angular_leafspot": 0.922, "anthracnose_fruit_rot": 0.833, "blossom_blight": 0.926, "gray_mold": 0.889, "leaf_spot": 0.918, "powdery_mildew_fruit": 0.687, "powdery_mildew_leaf": 0.916}, "bench_fp_035": 0.041}
print(f"\nDibanding {REF['nama']}:")
print(f"  test mAP50     {REF['test_map50']:.3f} -> {summary['test_map50']:.3f} ({summary['test_map50'] - REF['test_map50']:+.3f})")
print(f"  test mAP50-95  {REF['test_map50_95']:.3f} -> {summary['test_map50_95']:.3f} ({summary['test_map50_95'] - REF['test_map50_95']:+.3f})")
for name, ref in REF["AP50"].items():
    if name in PER_CLASS:
        print(f"  AP50 {name:24s} {ref:.3f} -> {PER_CLASS[name]['AP50']:.3f} ({PER_CLASS[name]['AP50'] - ref:+.3f})")
if BENCH_FP:
    print(f"  alarm palsu patokan (conf >= 0.35)  {REF['bench_fp_035']:.1%} -> {BENCH_FP['0.35']:.1%}")

In [ ]:
# 9) Kurva training dan confusion matrix
from IPython.display import Image, display

for f in [BEST.parent.parent / "results.png", Path(metrics.save_dir) / "confusion_matrix_normalized.png"]:
    if f.exists():
        print(f.name)
        display(Image(filename=str(f)))

In [ ]:
# 10) Cek visual: prediksi pada 8 gambar test acak (bukan negatif); judul = nama file (kelas sebenarnya ada di awal nama)
import random
import matplotlib.pyplot as plt

test_imgs = sorted(p for p in SPLITS["test"] if not p.name.startswith("neg_"))
sample = random.Random(0).sample(test_imgs, 8)
results = best.predict([str(p) for p in sample], imgsz=IMGSZ, conf=0.35, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, p, r in zip(axes.ravel(), sample, results):
    ax.imshow(r.plot()[:, :, ::-1])  # plot() menghasilkan BGR -> balik ke RGB
    ax.set_title(p.stem, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 11) Salin bobot ke /kaggle/working supaya mudah diunduh dari panel Output
import shutil
shutil.copy(BEST, "best.pt")
print("Tersimpan:", Path("best.pt").resolve(), f"({Path('best.pt').stat().st_size / 1e6:.1f} MB)")

## Setelah training

1. Unduh `best.pt` (panel **Output** di sisi kanan, atau tab Output setelah Save & Run All).
2. Taruh di folder proyek Anda, lalu jalankan `realtime.py` di laptop:

```
pip install -U ultralytics paho-mqtt     # paho-mqtt hanya jika memakai MQTT
python realtime.py --weights best.pt --source 0                                # webcam laptop
python realtime.py --weights best.pt --source http://IP-ESP32:81/stream --fps 1 --no-show
```

Kalau daftar kelas model memuat `strawberry_leaf` (hanya jika paket tambahan dibuat dengan `--leaf-mode class`), `realtime.py` tetap menggambarnya tetapi tidak menjadikannya alarm penyakit.

Bandingkan angka alarm palsu di sel 6 dan metrik per kelas di sel 5 antar percobaan. Kalau masih tinggi, tambah gambar negatif dari kamera dan lokasi yang akan dipakai, lalu latih ulang. Foto dari ESP32-CAM sendiri juga perlu dipakai untuk fine-tune, karena kualitas gambarnya berbeda dari foto dataset.

## Catatan percobaan

| Run | Perubahan | Test mAP50 | AP50 powdery mildew buah | Alarm palsu strawberry sehat (conf ≥ 0,35) |
|---|---|---|---|---|
| 1 | pembagian bawaan | 0,818 (test lama) | 0,73 | 2,3% |
| 2 | pembagian bersih + oversampling kelas langka | 0,870 | 0,69 | 4,1% |
| 3 | + 300 negatif strawberry matang/mentah (`strawberry_extra_v2`) | (isi) | (isi) | (isi) |

Angka run 2 di tabel dan di sel 8 adalah hasil uji ulang pada test set baru (75 negatif baru ikut di test, jadi sedikit berbeda dari laporan run 2). Run 3 cukup dijalankan tanpa mengubah pengaturan apa pun. Yang dicari: alarm palsu di sel 7 turun, sementara AP50 powdery mildew buah di sel 5 tidak turun banyak. Kalau recall-nya turun tajam, negatif strawberry terlalu banyak; kurangi lewat `--n-new` di `extend_extra.py`.

Patokan di sel 7 hanya foto dari internet. Ukuran sebenarnya tetap foto dari kamera yang akan dipakai di kebun.